# Project: Catan AI
***Dev log***
### Author: Alexander Makedonski

**Date:** 2026-03-13  
**Start time:** 16:00  
**End time:** 17:00  
**Session goal:** Setup project. Run a game  

Cloned catanatron (https://github.com/bcollazo/catanatron) as submodule. Added .gitignore.
Created venv: `py -m venv`. Then activate with: `.\venv\Scripts\Activate.ps1` if windows or `./venv/Scripts/activate` if linux. Installed developer dependencies: `pip install -e ./catanatron[web,gym,dev]`.
Ran a small simulation to check working dependencies.



**Date:** 2026-03-16  
**Start time:** 09:30  
**End time:** 12:00  
**Session goal:** Read documentation about catanatron. Find out a way to use for AI training  

The gym offers the possibility to generate data from games and use that as a point of study.
From what I gathered by skimming the entire documentation on https://docs.catanatron.com information from a game comes in 2 usable forms:  
1. One is the default observation of the board that is returned by `env.step()`/`env.reset()`. That observation is just a raw array of size `194 * N + 226` where `N` is the number of players on board  (source: https://catanatron.readthedocs.io/en/latest/catanatron.gym.envs.html#id2). 
2. The other one can be obtained with `catanatron.gym.board_tensor_features.create_board_tensor(game: Game, p0_color: Color, channels_first=False)` - A tensor of shape `(WIDTH=21, HEIGHT=11, CHANNELS=2*N+12)`, where `N` is the number of player in the game.

Thoughts: Skimmed over both variants to check what information is contained. Both seem appropriate, however variant 2:
- It contains a better representation of the board, but it alone is not enough to describe the current situation as it is missing information as IS_MOVING_ROBBER or IS_DISCARDING fields, that are contained in the first variant.
- It contains a lot of redundant information. A lot of the fields will stay 0 for the entire duration of the game. For example the tensor cell value that represents a road has N channels for every player, despite only one player ever being able to take control of it for the duration of the game.

The first arr will have to be used for sure (or some other methods) to get the needed information to feed the algorithm to play the game. Or maybe it can be used as enchancement for the second array, by adding more values to the tensor. But this might explode the space of possible inputs. Maybe can be used for building own observation features.

Notes: Seems like classic env is CatanatronEnv's default play is 1v1 with random player. (source: https://catanatron.readthedocs.io/en/latest/catanatron.gym.envs.html#module-catanatron.gym.envs.catanatron_env)

**Date:** 2026-03-25  
**Start time:** 15:00  
**End time:** 16:00  
**Session goal:** Understand the catanatron gym environment configuration settings. Try to setup DQN model to learn on 4 player map with randoms. Maybe graph the success rate.

Thoughts:
Ok, so one has to think how they will actually train the DQN. The DQN requires memory of previous (recent) experiences to be fed to the AI with a reward value. However it is hard to accertain what that reward value will be during the game and the moves.
1. Rewarding at the end of the game. I can map the not possible in the current state actions the DQN tries to make as bad, and some big negative value. That might not be a good idea, since that might discourage it from making moves that are possible in rare occasions. So it shouldn't be a big negative reward, but something small, like -0.001. But what about actions that it can make? One way is to map them to a reward value after the game is finished. If the DQN wins, it maps them to possitive values, negative otherwise - That doesn't sound good, because the agent doesn't know which action actually led it to winning the game so it might lead to something unstable.
2. Rewarding troughout the game. Give points for gaining resources, take points away for discarding, give points for building, getting longest road, largest army. Small rewards for small actions. And one big reward in the end for the win.

Actions:
Created some helper functions. More experiments required. 

**Date:** 2026-03-26  
**Start time:** 11:00  
**End time:** 14:30  
**Session goal:** Finished the setup of the DQN model.  

Thoughts:
Seems like yesterday there wasn't enough time to implement DQN, so I will try to do it now.
Actions:
Found a way to extract all the features of the game and make a named map for the observation. Create helpers with cache. Craeted reward function, memory for the DQN, the DQN itself and select_action function that masks unplayable actions. Created the q function for backprop. Needs testing and graphing to see how it works.


**Date:** 2026-04-18  
**Start time:** 14:30  
**End time:** 15:30  
**Session goal:** Run games with the DQN model. Excract some statistics for visualisation.  

Thoughts:
After making a  way to collect some information, I ran 2K games in about an hour. Which is really slow. Will have to find a way to upgrade the speed of the training. Also coded some way to extract info after the end of the game to see if the model is doing what it is supposed to do. Will have to update the reward function prbobably. Also maybe tweak the model and start smaller? Maybe even increase memory.
Actions:
Created some util functions to extract stats. Ran 2K games.

**Date:** 2026-04-19  
**Start time:** 14:30  
**End time:** 16:30  
**Session goal:** Optimize training. Try changing some settings.  

Thoughts:
Optimized dqn training. Now can do 1000 games in 30 minutes. Sadly training is not doing good, the model does not learn how to play the game, despite playing against random palyers. Changed weights of the mode to smaller model. Still not doing good. Started logging length of games, total_loss and reward for the game. Seems like dqn does fare well in this environment. Maybe because rewards are too sparse. Maybe I should try using the defalt reward function provided by the catanatron gym, since I am using my own now. Will probably try to define some unit tests to see if all the information saved for the model to train is acceptable.
Actions:
Optimized model training, by storing tensors directly in memory instead of transforming lists to tensors anytime they are batched for training. Also made tensors for actions masks for easier calculation later instead of an entire for cycle. Ran more trainings.

**Date:** 2026-04-21  
**Start time:** 14:00  
**End time:** 17:00  
**Session goal:** Change settings, try to achieve better training results.  

Thoughts:
Honestly, idk why DQN is not reaching at least somewhat good results. It appears to achieve randomness. Will probably increase memory size, play a bit with epsilon and other settings to see if anything changes. Maybe make the model more complex, since the game is really complex too.
Actions:
Optimized model training, by storing tensors directly in memory instead of transforming lists to tensors anytime they are batched for training. Also made tensors for actions masks for easier calculation later instead of an entire for cycle. Ran more trainings.